In [1]:
import numpy as np
from astropy.io import fits

In [2]:
opencov = fits.open('../data/covariance/xip_xim_map3_covariance_8ARCMINCUT_31Oct24.fits')
cov_matrix = opencov[1].data
opendata_2pt = fits.open('../data/dv/sim_2pt-NLA-cosmoCosmogrid-04Nov24.fits')
opendata_map3 = fits.open('../data/dv/sim_map3_emu-NLA-cosmoCosmogrid-02DECEMBER_WITH_COV.fits')

data_vector = np.zeros(480)
for i in range(480):
    if i < 200:
        data_vector[i] = opendata_2pt[2].data[i][3]
    elif i < 400:
        data_vector[i] = opendata_2pt[3].data[i-200][3]
    else:
        data_vector[i] = opendata_map3[1].data[i-400][6]

In [3]:
#Now we geneare noisy dv with the compressed 2pt function

transform = np.loadtxt("../data/moped/moped-moped-compress-2pt.txt")
dv2pt = data_vector[:400]
scale_cuts_xip = np.array([20,21,22,23,40,41,42,43,60,61,62,80,81,82,83,100,101,102,103,120,121,122,123,140,141,142,143,160,161,162,163,180,181,182])
scale_cuts_xim = np.concatenate((np.arange(200,210), np.arange(220,234),np.arange(240,254), np.arange(260,273), np.arange(280,294), np.arange(300,315), np.arange(320,335), np.arange(340,355), np.arange(360,375), np.arange(380,394)))
scale_cuts = np.concatenate((scale_cuts_xip, scale_cuts_xim))
scalecut_dv2pt = np.delete(dv2pt, scale_cuts)
compressed_dv = np.zeros(96)
compressed_dv[:16] = np.dot(transform.T, scalecut_dv2pt)
compressed_dv[16:] = data_vector[400:]

In [4]:
#Now we generate the transformed cov

transform_joint = np.zeros((307,96))
transform_joint[:227,:16] = transform
transform_joint[227:,16:] = np.eye(80)

cov_cut = np.delete(np.delete(cov_matrix, scale_cuts, axis=0), scale_cuts, axis=1)

compressed_cov = np.dot(transform_joint.T, np.dot(cov_cut, transform_joint))

In [5]:
# Assume `covariance_matrix` is your covariance matrix (n x n)
# and `data_vector_noiseless` is your noiseless data vector (n,)

# Step 1: Cholesky decomposition
L = np.linalg.cholesky(compressed_cov)

# Step 2: Generate a single noisy data vector
def generate_noisy_vector(data_vector_noiseless, L):
    # Generate a vector of standard normal random values
    z = np.random.normal(0, 1, size=data_vector_noiseless.shape)
    # Create noise using L and z
    noise = np.dot(L, z)
    # Add noise to the noiseless data vector
    data_vector_noisy = data_vector_noiseless + noise
    return data_vector_noisy

# Generate multiple noisy vectors if needed
num_realizations = 10000  # or however many you need
noisy_data_vector_cholesky = np.array([generate_noisy_vector(compressed_dv, L) for _ in range(num_realizations)])

In [9]:
print(compressed_dv)
print(noisy_data_vector_cholesky[7])

[ 2.88484988e+01 -7.82943158e-01 -1.07136246e+00 -1.94632179e+00
  2.74459299e+00  2.34686604e+00  4.90283124e+00  1.77902325e+00
 -2.90550780e-01  1.41830653e+00  3.91877835e-01  1.21581517e+00
  6.33226921e-01  8.99923992e-02 -2.99623315e-01  3.57339053e-02
  4.33217665e-10  2.15148297e-10  5.07796791e-11  6.90740037e-12
  6.24536796e-10  2.95340393e-10  6.81780539e-11  9.20986462e-12
  7.48077386e-10  3.45168784e-10  7.89529912e-11  1.06501409e-11
  7.83949741e-10  3.59459536e-10  8.20521147e-11  1.10667717e-11
  9.44773303e-10  4.24022048e-10  9.58713683e-11  1.29024335e-11
  1.16793713e-09  5.10485023e-10  1.14554201e-10  1.54320519e-11
  1.23528678e-09  5.36353131e-10  1.20189772e-10  1.62014728e-11
  1.48040394e-09  6.29513656e-10  1.40494614e-10  1.89859448e-11
  1.57917861e-09  6.67046077e-10  1.48804018e-10  2.01368736e-11
  1.69177748e-09  7.10134656e-10  1.58464973e-10  2.14819232e-11
  1.51724693e-09  6.45249692e-10  1.43387223e-10  1.93132975e-11
  1.95530017e-09  8.09495

In [8]:
empirical_cov_matrix = np.cov(noisy_data_vector_cholesky, rowvar=False)
frobenius_diff = np.linalg.norm(empirical_cov_matrix[16:,:][:,16:] - compressed_cov[16:,:][:,16:], ord='fro') / np.linalg.norm(compressed_cov[16:,:][:,16:], ord='fro')
print(frobenius_diff)

0.03598154281095742


In [19]:
def save_noisy_compressed_realization(number):

    opendata_map3 = fits.open('../data/dv/sim_map3_emu-NLA-cosmoCosmogrid-02DECEMBER_WITH_COV.fits')
 
    cut_compressed_2pt = noisy_data_vector_cholesky[number][:16]
    for i in range(16,96):
        opendata_map3[1].data[i-16][6] = noisy_data_vector_cholesky[number][i]

    noisy_map3 = '../data/dv/noisy_realizations/december/sim_map3-NLA-cosmoCosmogrid-0'+str(number+1)+'.fits'
    opendata_map3.writeto(noisy_map3) 
    
    np.savetxt('../data/dv/compressed-moped/december/sim-2pt-noisy-compressed-0'+str(number+1)+'.txt',cut_compressed_2pt)

In [20]:
for j in range(9,50):
    save_noisy_compressed_realization(j)

In [ ]:
import random

# List of numbers to permute
numbers = [1, 2, 3]

# Generate five random permutations with repetition
random_permutations = [random.sample(numbers, len(numbers)) for _ in range(500)]

# Print the random permutations
for perm in random_permutations[27:32]:
    print(perm)